# EDA — Datos SIVIGILA Dengue Colombia 2007–2024

**Proyecto:** Sistema de Alerta Temprana de Dengue (SAT-Dengue)  
**Grupo:** 11 — MAIA PDS, Universidad de los Andes  
**Fuente:** SIVIGILA / INS — Notificaciones individuales de dengue (evento 210)  

---

Este notebook carga los 18 archivos xlsx (uno por año), los une en un único DataFrame,
analiza su calidad y estructura, y produce los insumos para el modelado.

## 0. Setup

In [ ]:
import warnings
import os
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:,.2f}'.format)

# ── Configuración de rutas ───────────────────────────────────────────────────
DATA_DIR   = Path('../data/raw/')        # carpeta con los xlsx anuales
OUT_DIR    = Path('../data/processed/')  # salida del CSV agregado
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Años de epidemia mayor — excluidos del período de referencia del corredor endémico
EPIDEMIC_YEARS = [2010, 2013, 2016, 2019]
REF_YEARS_START = 2007
REF_YEARS_END   = 2019

# Columnas mínimas necesarias (usar para reducir RAM si es necesario)
COLUMNS_TO_KEEP = [
    'CONSECUTIVE', 'FEC_NOT', 'SEMANA', 'ANO',
    'COD_MUN_O', 'Municipio_ocurrencia', 'Departamento_ocurrencia', 'COD_DPTO_O',
    'TIP_CAS', 'confirmados', 'PAC_HOS', 'CON_FIN',
    'EDAD', 'UNI_MED', 'SEXO'
]

print('Setup OK')

## 1. Carga de datos

> **Nota de memoria:** El archivo de 2024 pesa ~191 MB en disco; el DataFrame
> completo de los 18 años puede ocupar entre **3 y 5 GB de RAM**. Si tienes menos de 8 GB
> disponibles, activa `USE_MINIMAL_COLS = True` para cargar solo las columnas en `COLUMNS_TO_KEEP`.

In [ ]:
USE_MINIMAL_COLS = False  # Cambiar a True si hay restricción de RAM

xlsx_files = sorted(DATA_DIR.glob('Datos_*_210.xlsx'))
print(f'Archivos encontrados: {len(xlsx_files)}')
for f in xlsx_files:
    print(f'  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)')

In [ ]:
dfs = []
for f in xlsx_files:
    year = int(f.stem.split('_')[1])  # extrae el año del nombre de archivo
    print(f'Cargando {f.name} ...', end=' ')
    kwargs = dict(engine='openpyxl')
    if USE_MINIMAL_COLS:
        kwargs['usecols'] = COLUMNS_TO_KEEP
    chunk = pd.read_excel(f, **kwargs)
    chunk['source_file'] = year
    dfs.append(chunk)
    print(f'{len(chunk):,} filas')

df = pd.concat(dfs, ignore_index=True)
del dfs

print(f'\nDataFrame consolidado: {df.shape[0]:,} filas × {df.shape[1]} columnas')
print(f'Uso de memoria: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

In [ ]:
df.head(3)

## 2. Estructura y calidad

In [ ]:
df.info(memory_usage='deep')

In [ ]:
# Valores nulos por columna (solo columnas con al menos un nulo)
null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(2)
missing = pd.DataFrame({'nulos': null_counts, 'pct': null_pct})
missing = missing[missing['nulos'] > 0].sort_values('pct', ascending=False)
print(f'Columnas con valores nulos: {len(missing)} de {df.shape[1]}')
display(missing)

In [ ]:
# Filas duplicadas
n_dup = df.duplicated().sum()
print(f'Filas duplicadas: {n_dup:,} ({n_dup/len(df)*100:.2f}%)')

In [ ]:
# Distribución por tipo de caso (TIP_CAS)
tip_labels = {
    1: 'Sospechoso', 2: 'Probable',
    3: 'Confirmado clínico', 4: 'Confirmado por lab', 5: 'Descartado'
}
tip_counts = df['TIP_CAS'].value_counts().sort_index()
tip_counts.index = [tip_labels.get(i, str(i)) for i in tip_counts.index]
display(tip_counts.to_frame('casos').assign(pct=lambda x: (x['casos']/len(df)*100).round(2)))

In [ ]:
# Condición final (CON_FIN)
fin_labels = {1: 'Vivo', 2: 'Muerto'}
fin_counts = df['CON_FIN'].value_counts().sort_index()
fin_counts.index = [fin_labels.get(i, str(i)) for i in fin_counts.index]
display(fin_counts.to_frame('casos'))

# Distribución por año
print('\nCasos por año (source_file):')
display(df['source_file'].value_counts().sort_index().to_frame('total_registros'))

## 3. Serie temporal de casos

In [ ]:
# Parsear fecha de notificación
df['FEC_NOT'] = pd.to_datetime(df['FEC_NOT'], errors='coerce')
print(f'Fechas no parseadas: {df["FEC_NOT"].isna().sum():,}')

# Flag de caso confirmado
df['es_confirmado'] = (
    df['TIP_CAS'].isin([3, 4]) | (df['confirmados'] == 1)
).astype(int)

In [ ]:
# Casos totales y confirmados por año
yearly = df.groupby('source_file').agg(
    total=('CONSECUTIVE', 'count'),
    confirmados=('es_confirmado', 'sum')
).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
x = yearly['source_file']
ax.bar(x, yearly['total'], color='#8FA3BE', label='Total registros')
ax.bar(x, yearly['confirmados'], color='#BE1D2B', label='Confirmados')
ax.set_title('Casos de dengue notificados por año — Colombia 2007–2024')
ax.set_xlabel('Año')
ax.set_ylabel('Número de casos')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Estacionalidad: casos por semana epidemiológica (acumulado todos los años)
weekly_season = df.groupby('SEMANA').size().reset_index(name='casos')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(weekly_season['SEMANA'], weekly_season['casos'],
        color='#1A7F37', linewidth=2)
ax.fill_between(weekly_season['SEMANA'], weekly_season['casos'],
                alpha=0.15, color='#1A7F37')
ax.set_title('Estacionalidad: casos acumulados por semana epidemiológica (2007–2024)')
ax.set_xlabel('Semana epidemiológica')
ax.set_ylabel('Casos acumulados (todos los años)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# Serie semanal completa 2007–2024
df_valid_date = df.dropna(subset=['FEC_NOT']).copy()
df_valid_date['year_week'] = df_valid_date['FEC_NOT'].dt.to_period('W')
weekly_ts = df_valid_date.groupby('year_week').size().reset_index(name='casos')
weekly_ts['year_week_dt'] = weekly_ts['year_week'].dt.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(weekly_ts['year_week_dt'], weekly_ts['casos'],
        color='#1654A2', linewidth=0.8, alpha=0.9)
# Marcar años epidémicos
for ey in EPIDEMIC_YEARS:
    ax.axvspan(pd.Timestamp(f'{ey}-01-01'), pd.Timestamp(f'{ey}-12-31'),
               color='#BE1D2B', alpha=0.08, label=str(ey) if ey == EPIDEMIC_YEARS[0] else '')
ax.set_title('Serie temporal semanal de dengue — Colombia 2007–2024\n(zonas rojas = años de epidemia mayor)')
ax.set_xlabel('Fecha')
ax.set_ylabel('Casos por semana')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.legend(['Años epidémicos'])
plt.tight_layout()
plt.show()

## 4. Análisis geográfico

In [ ]:
# Top 20 municipios por total de casos
top_muni = (
    df.groupby(['COD_MUN_O', 'Municipio_ocurrencia', 'Departamento_ocurrencia'])
    .size()
    .reset_index(name='casos')
    .sort_values('casos', ascending=False)
    .head(20)
)
top_muni['label'] = top_muni['Municipio_ocurrencia'] + '\n(' + top_muni['Departamento_ocurrencia'] + ')'

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(top_muni['label'][::-1], top_muni['casos'][::-1], color='#1654A2')
ax.set_title('Top 20 municipios con más casos de dengue (2007–2024)')
ax.set_xlabel('Total de casos')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 departamentos
top_dpto = (
    df.groupby('Departamento_ocurrencia')
    .size()
    .reset_index(name='casos')
    .sort_values('casos', ascending=False)
)
top_dpto['pct'] = (top_dpto['casos'] / top_dpto['casos'].sum() * 100).round(2)
print('Top 10 departamentos:')
display(top_dpto.head(10))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_dpto['Departamento_ocurrencia'][:10][::-1],
        top_dpto['casos'][:10][::-1], color='#0E1E3C')
ax.set_title('Top 10 departamentos por casos de dengue (2007–2024)')
ax.set_xlabel('Total de casos')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
plt.tight_layout()
plt.show()

In [ ]:
# Municipios con datos suficientes para el modelo
# Criterio D4 del proyecto: ≥10 años con casos Y ≥200 casos acumulados
muni_years = (
    df[df['source_file'].between(2007, 2024)]
    .groupby('COD_MUN_O')
    .agg(
        total_casos=('CONSECUTIVE', 'count'),
        años_con_casos=('source_file', 'nunique'),
        nombre=('Municipio_ocurrencia', 'first'),
        departamento=('Departamento_ocurrencia', 'first')
    )
    .reset_index()
)

n_total_muni    = muni_years['COD_MUN_O'].nunique()
n_endemic_muni  = muni_years[
    (muni_years['años_con_casos'] >= 10) & (muni_years['total_casos'] >= 200)
].shape[0]

print(f'Municipios únicos con al menos 1 caso: {n_total_muni:,}')
print(f'Municipios endémicos (≥10 años con casos Y ≥200 casos): {n_endemic_muni:,}')
print(f'  → Estos son los municipios que entrará al modelo.')

## 5. Corredor endémico (metodología del proyecto)

El corredor endémico se calcula sobre el período de referencia **2007–2019**,
excluyendo los años de epidemia mayor definidos en `EPIDEMIC_YEARS`.
Para cada semana epidemiológica se calculan los percentiles P25 / mediana (P50) / P75
sobre los años de referencia.

In [ ]:
# Agregar primero a conteo semanal por municipio
weekly_muni = (
    df.groupby(['COD_MUN_O', 'Municipio_ocurrencia', 'Departamento_ocurrencia', 'ANO', 'SEMANA'])
    .size()
    .reset_index(name='casos')
)

# Datos de referencia: 2007-2019 excluyendo años epidémicos
ref = weekly_muni[
    (weekly_muni['ANO'] >= REF_YEARS_START) &
    (weekly_muni['ANO'] <= REF_YEARS_END) &
    (~weekly_muni['ANO'].isin(EPIDEMIC_YEARS))
].copy()

print(f'Años de referencia usados: {sorted(ref["ANO"].unique())}')

In [ ]:
# Corredor endémico NACIONAL (todos los municipios sumados por semana)
national_weekly = ref.groupby(['ANO', 'SEMANA'])['casos'].sum().reset_index()
national_channel = national_weekly.groupby('SEMANA')['casos'].agg(
    P25=lambda x: x.quantile(0.25),
    P50='median',
    P75=lambda x: x.quantile(0.75)
).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(national_channel['SEMANA'], national_channel['P25'], national_channel['P75'],
                alpha=0.25, color='#1A7F37', label='Banda P25–P75')
ax.plot(national_channel['SEMANA'], national_channel['P50'],
        color='#1A7F37', linewidth=2, label='Mediana (P50)')
ax.plot(national_channel['SEMANA'], national_channel['P25'],
        color='#1A7F37', linewidth=1, linestyle='--', alpha=0.7)
ax.plot(national_channel['SEMANA'], national_channel['P75'],
        color='#BE1D2B', linewidth=1, linestyle='--', alpha=0.7, label='P75 (umbral alerta)')
ax.set_title('Corredor endémico nacional — Período de referencia 2007–2019\n(excluyendo años epidémicos)')
ax.set_xlabel('Semana epidemiológica')
ax.set_ylabel('Casos semanales')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'{v:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Corredor endémico para los top 5 municipios más endémicos
top5_muni = (
    muni_years[
        (muni_years['años_con_casos'] >= 10) & (muni_years['total_casos'] >= 200)
    ]
    .sort_values('total_casos', ascending=False)
    .head(5)['COD_MUN_O'].tolist()
)

fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=False)
for ax, cod in zip(axes, top5_muni):
    muni_ref = ref[ref['COD_MUN_O'] == cod]
    nombre   = muni_ref['Municipio_ocurrencia'].iloc[0] if len(muni_ref) > 0 else cod
    channel  = muni_ref.groupby('SEMANA')['casos'].agg(
        P25=lambda x: x.quantile(0.25),
        P50='median',
        P75=lambda x: x.quantile(0.75)
    ).reset_index()

    ax.fill_between(channel['SEMANA'], channel['P25'], channel['P75'],
                    alpha=0.25, color='#1A7F37')
    ax.plot(channel['SEMANA'], channel['P50'], color='#1A7F37', linewidth=1.5)
    ax.plot(channel['SEMANA'], channel['P75'], color='#BE1D2B',
            linewidth=1, linestyle='--', alpha=0.8)
    ax.set_title(nombre, fontsize=9)
    ax.set_xlabel('Semana')
    ax.set_ylabel('Casos')

fig.suptitle('Corredor endémico — Top 5 municipios más endémicos', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 6. Preparación para modelado

Agrega las notificaciones individuales a conteos semanales por municipio
y guarda el CSV procesado en `../data/processed/`.

In [ ]:
# Tabla agregada: una fila = un municipio × una semana epidemiológica × un año
df_semanal = (
    df.groupby([
        'COD_MUN_O', 'Municipio_ocurrencia', 'Departamento_ocurrencia',
        'COD_DPTO_O', 'ANO', 'SEMANA'
    ])
    .size()
    .reset_index(name='casos')
    .sort_values(['COD_MUN_O', 'ANO', 'SEMANA'])
)

print(f'Tabla agregada: {df_semanal.shape[0]:,} filas × {df_semanal.shape[1]} columnas')
df_semanal.head()

In [ ]:
# Municipios con datos suficientes para el modelo (criterio D4)
muni_stats = (
    df_semanal.groupby('COD_MUN_O')
    .agg(
        nombre=('Municipio_ocurrencia', 'first'),
        departamento=('Departamento_ocurrencia', 'first'),
        total_casos=('casos', 'sum'),
        años_con_casos=('ANO', 'nunique')
    )
    .reset_index()
)

endemicos = muni_stats[
    (muni_stats['años_con_casos'] >= 10) & (muni_stats['total_casos'] >= 200)
].sort_values('total_casos', ascending=False)

print(f'Municipios que entran al modelo: {len(endemicos):,}')
display(endemicos.head(20))

In [ ]:
# Guardar CSV procesado
out_path = OUT_DIR / 'dengue_semanal_municipio.csv'
df_semanal.to_csv(out_path, index=False, encoding='utf-8')
print(f'CSV guardado en: {out_path}')
print(f'Tamaño: {out_path.stat().st_size / 1e6:.1f} MB')

# También guardar la lista de municipios endémicos
endemicos_path = OUT_DIR / 'municipios_endemicos.csv'
endemicos.to_csv(endemicos_path, index=False, encoding='utf-8')
print(f'Lista de municipios endémicos guardada en: {endemicos_path}')

---

## Resumen

| Métrica | Valor |
|---|---|
| Años cubiertos | 2007–2024 |
| Estructura | Idéntica en todos los archivos (69 columnas) |
| Período de referencia del corredor endémico | 2007–2019 excl. años epidémicos |
| Años epidémicos excluidos | 2010, 2013, 2016, 2019 |
| Umbral de exceso (D2) | Casos > P75 del corredor endémico |
| Municipios que entran al modelo (D4) | Ver celda anterior |
| CSV procesado | `../data/processed/dengue_semanal_municipio.csv` |